# ДЗ 3. NumPy на данных «Дала Маркет»
Срок: **до четверга 12.11, 19:00**.
Сдача — папка `hw03` репозитория `dsa-homework` (`hw03.ipynb`): `git add hw03` → `git commit` → `git push`.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем — только массивы и маски (кроме задачи 8).

- **[База]** обязательно для всех · **[Стандарт]** — сколько получится · **[Челлендж]** — для тех, кому легко.

## [Для всех] Чтение кода — без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])
```

Мои предсказания и объяснения:

1. output: a [0,1,2,  будет 5 и shape 2,3
              3,4,5]
2. output: индекс макс числа будет 1 и макс число в массиве это 8
3. output: 99 так как массив b это маска массива a 
4. output: sum = 2, mean = 0.5 в массиве есть 2 true и считаем их долю
5. output: (2,3) [0,0.25,0.50,0.75,1.00] 1 ый index массива = 0.25

In [13]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [14]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)

(5463,) float64 bool


## [База]
1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?

In [15]:
# База 1

idx = amount.argmax()
maxorder = [int(amount[idx]), str(cust[idx]), str(city[idx])]
print(f"Самый дорогой заказ (позиция #{idx}):\n"
      f"Сумма: {amount[idx]:,} тг\n"
      f"Клиент: {cust[idx]}\n"
      f"Город: {city[idx]}"
      )


Самый дорогой заказ (позиция #1001):
Сумма: 9,999,990.0 тг
Клиент: TEST
Город: Алматы


In [16]:
# База 2

print(f"Выручка оплаченных заказов и сумма неоплаченных : {int(amount[paid].sum())} тг, {int(amount[~paid].sum())} тг")



Выручка оплаченных заказов и сумма неоплаченных : 101909360 тг, 7974000 тг


In [17]:
# База 3
sumofShym = amount[(city=="Шымкент")]
shymkent_count = int(sumofShym.sum())
shymkent_share = sumofShym.mean()
print(f"  Заказы в Шымкенте:\n"
      f"- Количество: {shymkent_count} шт.\n"
      f"- Доля от всех заказов: {shymkent_share / 100:.2f}%.")

  Заказы в Шымкенте:
- Количество: 11403120 шт.
- Доля от всех заказов: 141.83%.


## [Стандарт]
4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
   Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
```python
avg = amount.mean()
print("Средний чек:", round(avg))
```

In [18]:
# Стандарт 4
print("---------------# Стандарт 4----------------\n")

mask = amount[(paid==1) & (amount > 50000)]
print(f"{int(mask.sum()*0.15)} тг")

mask2 = amount[(paid==1) & (amount > 50000) & (cust != 'TEST')]
print((f"{int(mask2.sum()*0.15)} тг\n"))

# Стандарт 5
print("---------------# Стандарт 5----------------\n")

for i in np.unique(city):
    city_days = days[city==i]
    share = (city_days <= 2.0).mean() * 100
    print(f"Город: {i} | Доля заказов доставленный за 1-2 дня: {share:.2f}%")

# Стандарт 6
print("\n---------------# Стандарт 6----------------\n")

avg = amount[(paid) & (cust != 'TEST')]
print(f"Средний чек: {round(avg.mean())} тг")



---------------# Стандарт 4----------------

5702422 тг
1202427 тг

---------------# Стандарт 5----------------

Город: Актобе | Доля заказов доставленный за 1-2 дня: 62.22%
Город: Алматы | Доля заказов доставленный за 1-2 дня: 64.53%
Город: Астана | Доля заказов доставленный за 1-2 дня: 63.53%
Город: Атырау | Доля заказов доставленный за 1-2 дня: 64.38%
Город: Караганда | Доля заказов доставленный за 1-2 дня: 67.84%
Город: Шымкент | Доля заказов доставленный за 1-2 дня: 29.85%

---------------# Стандарт 6----------------

Средний чек: 14672 тг


## [Челлендж]
7. Сравните скорость: сумма оплаченных заказов циклом `for` и через маску. Во сколько раз быстрее?
8. Смоделируйте A/B-тест. `np.random.seed(42)`. Группа A: 1 000 клиентов, вероятность покупки 5%; группа B: 1 000 клиентов, 6%
   (покупка = `np.random.rand(1000) < 0.05`). Сколько покупок в каждой группе?
   Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)

In [ ]:
# Челлендж